# Bahdanau Attention Mechanism - Simplified Tutorial

This notebook provides a simplified implementation of the Bahdanau Attention mechanism, originally proposed in the paper "Neural Machine Translation by Jointly Learning to Align and Translate" (2014).

## What is Attention?

In sequence-to-sequence models (like translation), attention allows the decoder to "focus" on different parts of the input sequence at each decoding step, rather than relying on a single fixed context vector.

## Key Concepts:
1. **Encoder hidden states**: Representations of each input word
2. **Decoder hidden state**: Current state of the decoder
3. **Attention scores**: How much to focus on each encoder state
4. **Context vector**: Weighted sum of encoder states

Let's build this step by step!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Step 1: Understanding the Attention Score Function

Bahdanau attention uses an **additive** (or concat) attention mechanism:

$$\text{score}(h_t, \bar{h}_s) = v^T \tanh(W_1 h_t + W_2 \bar{h}_s)$$

Where:
- $h_t$ is the decoder hidden state at time $t$
- $\bar{h}_s$ is the encoder hidden state at position $s$
- $W_1, W_2, v$ are learnable parameters

In [ ]:
class BahdanauAttention(nn.Module):
    """
    Bahdanau Attention (Additive Attention)
    """
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.hidden_size = hidden_size
        
        # Learnable parameters
        self.W1 = nn.Linear(hidden_size, hidden_size, bias=False)  # For decoder state
        self.W2 = nn.Linear(hidden_size, hidden_size, bias=False)  # For encoder states
        self.V = nn.Linear(hidden_size, 1, bias=False)              # Output projection
    
    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: (batch_size, hidden_size) - current decoder state
            encoder_outputs: (batch_size, seq_len, hidden_size) - all encoder states
        
        Returns:
            context: (batch_size, hidden_size) - weighted sum of encoder outputs
            attention_weights: (batch_size, seq_len) - attention distribution
        """
        batch_size, seq_len, hidden_size = encoder_outputs.size()
        
        # Step 1: Expand decoder hidden to match encoder outputs
        # (batch_size, hidden_size) -> (batch_size, 1, hidden_size) -> (batch_size, seq_len, hidden_size)
        decoder_hidden_expanded = decoder_hidden.unsqueeze(1).expand(-1, seq_len, -1)
        
        # Step 2: Calculate attention scores
        # score = V * tanh(W1(decoder) + W2(encoder))
        energy = torch.tanh(self.W1(decoder_hidden_expanded) + self.W2(encoder_outputs))
        scores = self.V(energy).squeeze(-1)  # (batch_size, seq_len)
        
        # Step 3: Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=1)  # (batch_size, seq_len)
        
        # Step 4: Calculate context vector as weighted sum
        # (batch_size, 1, seq_len) @ (batch_size, seq_len, hidden_size) -> (batch_size, 1, hidden_size)
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs)
        context = context.squeeze(1)  # (batch_size, hidden_size)
        
        return context, attention_weights

## Step 2: Simple Example with Dummy Data

Let's create a simple example to see how attention works.

In [ ]:
# Hyperparameters
batch_size = 2
seq_len = 5  # Input sequence length (e.g., "Hello how are you ?")
hidden_size = 8

# Create dummy encoder outputs (simulating encoded input sequence)
encoder_outputs = torch.randn(batch_size, seq_len, hidden_size)

# Create dummy decoder hidden state (current state of decoder)
decoder_hidden = torch.randn(batch_size, hidden_size)

print(f"Encoder outputs shape: {encoder_outputs.shape}")
print(f"Decoder hidden shape: {decoder_hidden.shape}")

In [ ]:
# Initialize attention mechanism
attention = BahdanauAttention(hidden_size)

# Apply attention
context, attention_weights = attention(decoder_hidden, encoder_outputs)

print(f"Context vector shape: {context.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"\nAttention weights for first example:\n{attention_weights[0]}")
print(f"\nSum of attention weights: {attention_weights[0].sum():.4f} (should be ~1.0)")

## Step 3: Visualizing Attention Weights

Let's visualize how the attention mechanism distributes its focus across the input sequence.

In [ ]:
def visualize_attention(attention_weights, source_words=None, target_word=None):
    """
    Visualize attention weights as a heatmap
    """
    plt.figure(figsize=(10, 3))
    
    # Convert to numpy
    weights = attention_weights.detach().numpy()
    
    # Create heatmap
    sns.heatmap(weights.reshape(1, -1), 
                annot=True, 
                fmt='.3f', 
                cmap='YlOrRd',
                cbar=True,
                xticklabels=source_words if source_words else range(len(weights)),
                yticklabels=[target_word] if target_word else ['Attention'])
    
    plt.xlabel('Source Sequence (Encoder States)')
    plt.ylabel('Target')
    plt.title('Attention Distribution')
    plt.tight_layout()
    plt.show()

# Visualize for first example
visualize_attention(attention_weights[0], 
                   source_words=['Word1', 'Word2', 'Word3', 'Word4', 'Word5'],
                   target_word='Decoder_t')

## Step 4: Simulating a Translation Example

Let's simulate a more realistic scenario where we translate "I love machine learning" to French.

In [ ]:
# Simulate a translation scenario
source_sentence = ["I", "love", "machine", "learning"]
target_sentence = ["J'", "adore", "l'apprentissage", "automatique"]

# Create encoder outputs for each source word
seq_len = len(source_sentence)
encoder_outputs = torch.randn(1, seq_len, hidden_size)  # batch_size = 1

# Simulate decoding steps
print("Simulating attention at each decoding step:\n")
all_attention_weights = []

for t, target_word in enumerate(target_sentence):
    # Create a different decoder hidden state for each step
    decoder_hidden = torch.randn(1, hidden_size)
    
    # Apply attention
    context, attn_weights = attention(decoder_hidden, encoder_outputs)
    all_attention_weights.append(attn_weights[0].detach().numpy())
    
    print(f"Target word: {target_word:20s} | Attention: {attn_weights[0].detach().numpy()}")

In [ ]:
# Visualize all attention weights as a matrix
def visualize_attention_matrix(attention_weights, source_words, target_words):
    """
    Visualize attention weights for all decoding steps
    """
    plt.figure(figsize=(10, 6))
    
    attention_matrix = np.array(attention_weights)
    
    sns.heatmap(attention_matrix,
                annot=True,
                fmt='.3f',
                cmap='YlOrRd',
                xticklabels=source_words,
                yticklabels=target_words,
                cbar_kws={'label': 'Attention Weight'})
    
    plt.xlabel('Source Words (English)', fontsize=12)
    plt.ylabel('Target Words (French)', fontsize=12)
    plt.title('Attention Alignment Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_attention_matrix(all_attention_weights, source_sentence, target_sentence)

## Step 5: Understanding the Attention Mechanism Step-by-Step

Let's break down what happens inside the attention mechanism with actual numbers.

In [ ]:
# Create a simple example with small dimensions for easy visualization
hidden_size_small = 3
seq_len_small = 4

# Simple encoder outputs and decoder hidden state
encoder_outputs_small = torch.randn(1, seq_len_small, hidden_size_small)
decoder_hidden_small = torch.randn(1, hidden_size_small)

print("Encoder outputs (4 words, 3 dimensions each):")
print(encoder_outputs_small[0])
print("\nDecoder hidden state:")
print(decoder_hidden_small[0])

In [ ]:
# Initialize small attention
attention_small = BahdanauAttention(hidden_size_small)

# Manually trace through the attention calculation
with torch.no_grad():
    # Step 1: Expand decoder hidden
    decoder_expanded = decoder_hidden_small.unsqueeze(1).expand(-1, seq_len_small, -1)
    print("Step 1 - Decoder hidden expanded to match sequence length:")
    print(f"Shape: {decoder_expanded.shape}")
    
    # Step 2: Calculate energy
    energy = torch.tanh(
        attention_small.W1(decoder_expanded) + 
        attention_small.W2(encoder_outputs_small)
    )
    print("\nStep 2 - Energy after tanh(W1*decoder + W2*encoder):")
    print(f"Shape: {energy.shape}")
    print(energy[0])
    
    # Step 3: Calculate scores
    scores = attention_small.V(energy).squeeze(-1)
    print("\nStep 3 - Scores (before softmax):")
    print(scores[0])
    
    # Step 4: Apply softmax
    attention_weights_small = F.softmax(scores, dim=1)
    print("\nStep 4 - Attention weights (after softmax):")
    print(attention_weights_small[0])
    print(f"Sum: {attention_weights_small[0].sum():.4f}")
    
    # Step 5: Calculate context
    context_small = torch.bmm(attention_weights_small.unsqueeze(1), encoder_outputs_small)
    context_small = context_small.squeeze(1)
    print("\nStep 5 - Context vector (weighted sum):")
    print(context_small[0])

## Step 6: Compare with Simple Average (No Attention)

Let's compare using attention vs. simply averaging all encoder outputs.

In [ ]:
# Context with attention
context_attention, attn_weights = attention_small(decoder_hidden_small, encoder_outputs_small)

# Context without attention (simple average)
context_average = encoder_outputs_small.mean(dim=1)

print("Context vector WITH attention:")
print(context_attention[0])
print(f"\nAttention weights: {attn_weights[0]}")
print("\n" + "="*50)
print("\nContext vector WITHOUT attention (simple average):")
print(context_average[0])
print("\nImplicit weights: [0.25, 0.25, 0.25, 0.25] (equal for all)")
print("\n" + "="*50)
print("\nDifference:")
print("Attention allows the model to focus on specific input positions!")
print("The decoder can 'look at' relevant parts of the input.")

## Key Takeaways

### 1. **Attention solves the bottleneck problem**
   - In vanilla seq2seq, the entire input is compressed into a single fixed-size vector
   - Attention allows the decoder to access all encoder hidden states

### 2. **Bahdanau attention is additive**
   - Uses a feedforward network to compute alignment scores
   - Formula: $\text{score}(h_t, \bar{h}_s) = v^T \tanh(W_1 h_t + W_2 \bar{h}_s)$
   - Contrast with dot-product attention: $\text{score}(h_t, \bar{h}_s) = h_t^T \bar{h}_s$

### 3. **The attention weights are interpretable**
   - They show which input words the model focuses on
   - Useful for understanding model behavior and debugging

### 4. **The context vector is dynamic**
   - Different at each decoding step
   - Adapts to what the decoder needs at each time step

## Exercise for Students

Try modifying the code to:

1. **Change the hidden size** - How does this affect the attention weights?
2. **Implement dot-product attention** - Compare with Bahdanau attention
3. **Add masking** - Prevent attention to padding tokens
4. **Visualize attention** on real translation pairs using a pre-trained model

### Bonus Challenge:
Implement the full seq2seq model with Bahdanau attention for a simple translation task!

## References

1. Bahdanau, D., Cho, K., & Bengio, Y. (2014). Neural Machine Translation by Jointly Learning to Align and Translate. arXiv:1409.0473
2. Luong, M. T., Pham, H., & Manning, C. D. (2015). Effective Approaches to Attention-based Neural Machine Translation. arXiv:1508.04025

---

**Happy Learning! 🎓**